SATD detection analysis for MAT


In [14]:
import pandas as pd
import os
# -----------------------------
# File paths
# -----------------------------
COMMENTS_FILE = "../data/comment.csv"
PRED_FILE = "../cache/output/detect/unique/detect_pretrained-MAT.csv"

OUTPUT_ANALYSIS = "../cache/output/analysis/MAT_satd_prediction_all.csv"
OUTPUT_TP = "../cache/output/analysis/MAT_satd_prediction_correct_tp.csv"
OUTPUT_FN = "../cache/output/analysis/MAT_satd_prediction_missed_fn.csv"
os.makedirs(os.path.dirname(OUTPUT_ANALYSIS), exist_ok=True)

# -----------------------------
# Load data
# -----------------------------
comments = pd.read_csv(COMMENTS_FILE)
pred = pd.read_csv(PRED_FILE)

# -----------------------------
# Merge predictions with ground truth
# -----------------------------
df = comments.merge(
    pred[['id', 'label_pred']],
    on="id",
    how="inner"
)

# -----------------------------
# Convert to binary SATD detection task
# -----------------------------
df["is_satd"] = df["satd"] == "yes"
df["pred_satd"] = df["label_pred"] == "yes"

# -----------------------------
# Compute confusion matrix label
# -----------------------------
df["confusion"] = "TN"

df.loc[(df.is_satd) & (df.pred_satd), "confusion"] = "TP"
df.loc[(df.is_satd) & (~df.pred_satd), "confusion"] = "FN"
df.loc[(~df.is_satd) & (df.pred_satd), "confusion"] = "FP"
df.loc[(~df.is_satd) & (~df.pred_satd), "confusion"] = "TN"

# -----------------------------
# SATD-only dataset
# -----------------------------
satd_df = df[df["is_satd"]].copy()

# -----------------------------
# Compute statistics per SATD type
# -----------------------------
stats = (
    satd_df
    .groupby(["type", "confusion"])
    .size()
    .unstack(fill_value=0)
)

# Ensure required columns exist
for c in ["TP", "FN"]:
    if c not in stats.columns:
        stats[c] = 0

# Compute totals and percentages
stats["total_satd"] = stats["TP"] + stats["FN"]
stats["correct"] = stats["TP"]
stats["failed"] = stats["FN"]

stats["correct_%"] = (stats["correct"] / stats["total_satd"] * 100).round(2)
stats["failed_%"] = (stats["failed"] / stats["total_satd"] * 100).round(2)

stats = stats.sort_values("total_satd", ascending=False)

print("\nSATD Detection Performance by Type\n")
print(stats)


# -----------------------------
# Export SATD-only CSV with confusion labels
# -----------------------------
output_cols = list(comments.columns) + ["label_pred", "confusion"]

satd_output = satd_df[output_cols]

satd_output.to_csv(
    OUTPUT_ANALYSIS,
    index=False
)


# -----------------------------
# Export correctly detected SATDs
# -----------------------------
satd_output[satd_output["confusion"] == "TP"].to_csv(
    OUTPUT_TP,
    index=False
)



# -----------------------------
# Export missed SATDs
# -----------------------------
satd_output[satd_output["confusion"] == "FN"].to_csv(
    OUTPUT_FN,
    index=False
)




SATD Detection Performance by Type

confusion             FN  TP  total_satd  correct  failed  correct_%  failed_%
type                                                                          
composite             12  12          24       12      12      50.00     50.00
defect                 4  16          20       16       4      80.00     20.00
how-to                 8  10          18       10       8      55.56     44.44
workaround            12   2          14        2      12      14.29     85.71
requirement            3  10          13       10       3      76.92     23.08
low-internal-quality   1   7           8        7       1      87.50     12.50
documentation          2   3           5        3       2      60.00     40.00
dependency             3   0           3        0       3       0.00    100.00
design                 1   2           3        2       1      66.67     33.33
partial-test           2   1           3        1       2      33.33     66.67
skip-test      

RQ3 False Negative (FN) Analysis

In [26]:
import pandas as pd
import os

INPUT_FILE = "../result/rq3/unique/detect_flan-t5-xxl-mat-0-shot.csv"
OUTPUT_FILE = "../cache/output/analysis/detect_flan-t5-xxl-mat-0-shot_random_fn.csv"

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)


# Load detection results
df = pd.read_csv(INPUT_FILE)

# Select False Negatives (SATD predicted as non-SATD)
fn_df = df[(df["label"] != "no") & (df["label_pred"] == "no")]
print(f"False Negatives: {len(fn_df)}/{len(df)}")

# For 95% confidence of ~24 samples is 23 so take the whole
IDEAL_SAMPLE_SIZE = len(fn_df)



# Random sample
sample_df = fn_df.sample(
    n=min(IDEAL_SAMPLE_SIZE, len(fn_df)),
    random_state=42
)

# Save to CSV
sample_df.to_csv(OUTPUT_FILE, index=False)

False Negatives: 24/6531


RQ4 False Positive (FP) Analysis

In [27]:
import pandas as pd
import os

INPUT_FILE = "../result/rq4/unique/detect_gpt-5-2-shot.csv"
OUTPUT_FILE = "../cache/output/analysis/detect_gpt-5-2-shot_random_fp.csv"
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Load detection results
df = pd.read_csv(INPUT_FILE)

# Select False Positives (non-SATD predicted as SATD)
fp_df = df[(df["label"] == "no") & (df["label_pred"] == "yes")]
print(f"False Positives: {len(fp_df)}/{len(df)}")

# For 95% confidence of 233 samples is 146
IDEAL_SAMPLE_SIZE = 146



# Random sample
sample_df = fp_df.sample(n=min(IDEAL_SAMPLE_SIZE, len(fp_df)), random_state=42)

# Save to CSV
sample_df.to_csv(OUTPUT_FILE, index=False)


False Positives: 233/6531
